In [ ]:
# Cell 0 — Environment setup and dataset path detection
import os
import pandas as pd
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    input_root = Path("/kaggle/input")

    # Find folder containing train/, test/, and submission.csv
    data_dir = None
    for check_dir in [input_root] + [d for d in input_root.rglob("*") if d.is_dir()]:
        if (
            (check_dir / "train").exists()
            and (check_dir / "test").exists()
            and (check_dir / "submission.csv").exists()
        ):
            data_dir = check_dir
            break

    if data_dir is None:
        raise FileNotFoundError(
            "No folder found containing train/, test/, and submission.csv "
            "under /kaggle/input/."
        )

    output_dir = Path("/kaggle/working")

else:
    # Local: search for folder with 'satria-data-bdc' in name
    current_dir = Path.cwd()
    data_dir = None
    check_dir = current_dir
    while check_dir != check_dir.parent:
        if "satria-data-bdc" in check_dir.name.lower():
            data_dir = check_dir
            break
        check_dir = check_dir.parent
    if data_dir is None:
        data_dir = current_dir
    output_dir = data_dir

train_dir       = data_dir / "train"
test_dir        = data_dir / "test"
submission_path = data_dir / "submission.csv"

assert train_dir.exists(),       f"train_dir not found: {train_dir}"
assert test_dir.exists(),        f"test_dir not found: {test_dir}"
assert submission_path.exists(), f"submission.csv not found: {submission_path}"

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"data_dir    : {data_dir}")
print(f"train_dir   : {train_dir}  ({len(list(train_dir.glob('*/*')))} files)")
print(f"test_dir    : {test_dir}  ({len(list(test_dir.glob('*')))} files)")
print(f"submission  : {submission_path}  ({submission_path.stat().st_size:,} bytes)")
print(f"output_dir  : {output_dir}")

In [ ]:
# Cell 0b — Generate duplicate and train-test overlap CSVs
import re
import hashlib
from collections import defaultdict

def compute_file_hash(file_path, chunk_size=8192):
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()

def hash_all(root_dirs):
    hash_map  = defaultdict(list)
    all_files = []
    exts      = {".jpg", ".jpeg", ".png"}
    for label, folder in root_dirs.items():
        folder = Path(folder)
        for fp in folder.rglob("*"):
            if fp.is_file() and fp.suffix.lower() in exts:
                h = compute_file_hash(fp)
                hash_map[h].append((label, str(fp)))
                all_files.append((label, fp))
    return hash_map, all_files

train_dirs = {
    "0_Recyclable": train_dir / "0_Recyclable",
    "1_Electronic": train_dir / "1_Electronic",
    "2_Organic":    train_dir / "2_Organic",
}
test_dir_map = {"test": test_dir}

print("Hashing train files...")
train_hashes, train_files = hash_all(train_dirs)
print("Hashing test files...")
test_hashes, _ = hash_all(test_dir_map)

# Part A: exact train-test overlap
overlap_rows = []
for h, train_entries in train_hashes.items():
    if h in test_hashes:
        for (train_label, train_path) in train_entries:
            for (_, test_path) in test_hashes[h]:
                overlap_rows.append({
                    "md5":            h,
                    "train_label":    train_label,
                    "train_path":     train_path,
                    "train_filename": Path(train_path).name,
                    "test_path":      test_path,
                    "test_id":        int(Path(test_path).stem),
                })
overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(output_dir / "train_test_overlap.csv", index=False)
print(f"[A] Train-test overlap: {len(overlap_df)} rows")

# Part B: exact duplicate groups within train
group_rows = []
group_id   = 0
for h, entries in train_hashes.items():
    if len(entries) > 1:
        group_id += 1
        labels_in_group = set(label for label, _ in entries)
        is_cross_class  = len(labels_in_group) > 1
        for (label, path) in entries:
            group_rows.append({
                "md5":                  h,
                "duplicate_group_id":   group_id,
                "label":                label,
                "filepath":             path,
                "filename":             Path(path).name,
                "is_cross_class_group": is_cross_class,
                "match_type":           "exact_md5",
            })
groups_df = pd.DataFrame(group_rows)
groups_df.to_csv(output_dir / "train_duplicate_groups.csv", index=False)
print(f"[A] Exact duplicate groups: {groups_df['duplicate_group_id'].nunique()} groups")

# Part C: near-duplicate candidates via copy filename pattern e.g. "file(1).jpg"
copy_pattern = re.compile(r"^(.*)\((\d+)\)(\.\w+)$")
name_to_path = {fp.name: (label, fp) for label, fp in train_files}
candidates   = []

for label, fp in train_files:
    m = copy_pattern.match(fp.name)
    if m:
        base_name = m.group(1) + m.group(3)
        if base_name in name_to_path:
            orig_label, orig_fp = name_to_path[base_name]
            h_copy = compute_file_hash(fp)
            h_orig = compute_file_hash(orig_fp)
            candidates.append({
                "file_a":    fp.name,
                "file_b":    base_name,
                "label_a":   label,
                "label_b":   orig_label,
                "md5_match": h_copy == h_orig,
                "path_a":    str(fp),
                "path_b":    str(orig_fp),
            })

cand_df     = pd.DataFrame(candidates)
near_dup_df = cand_df[~cand_df["md5_match"]].copy()
near_dup_df.to_csv(output_dir / "near_duplicate_candidates.csv", index=False)
print(f"[C] Near-duplicate candidates: {len(near_dup_df)} pairs")

In [ ]:
# Cell 0c — Load duplicate CSVs from output_dir
overlap_df  = pd.read_csv(output_dir / "train_test_overlap.csv")
groups_df   = pd.read_csv(output_dir / "train_duplicate_groups.csv")
near_dup_df = pd.read_csv(output_dir / "near_duplicate_candidates.csv")

print(f"overlap_df  : {overlap_df.shape}")
print(f"groups_df   : {groups_df.shape}")
print(f"near_dup_df : {near_dup_df.shape}")

In [ ]:
# Cell 1 — Build train file list with labels
label_map = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}

records = []
for folder in train_dir.iterdir():
    if not folder.is_dir():
        continue
    if folder.name not in label_map:
        print(f"Warning: unexpected folder '{folder.name}' — skipped")
        continue
    label_id = label_map[folder.name]
    for fp in folder.glob("*"):
        if fp.is_file():
            records.append({
                "filename":     fp.name,
                "filepath":     str(fp),
                "label":        label_id,
                "label_folder": folder.name,
            })

train_master = pd.DataFrame(records)
print(train_master.shape)
train_master.head()

In [ ]:
# Cell 2 — Merge exact duplicate group IDs into train_master
train_master["duplicate_group_id"] = range(len(train_master))

filename_to_group = dict(zip(groups_df["filename"], groups_df["duplicate_group_id"]))
offset = train_master["duplicate_group_id"].max() + 1
filename_to_group = {k: v + offset for k, v in filename_to_group.items()}

train_master["duplicate_group_id"] = train_master.apply(
    lambda row: filename_to_group.get(row["filename"], row["duplicate_group_id"]),
    axis=1,
)

n_files           = len(train_master)
n_exact_dup_files = groups_df["filename"].nunique()
expected_unique   = n_files - n_exact_dup_files + groups_df["duplicate_group_id"].nunique()

assert train_master["duplicate_group_id"].nunique() == expected_unique, (
    f"Group collapse mismatch: got {train_master['duplicate_group_id'].nunique()}, "
    f"expected {expected_unique}"
)
print("Unique groups after exact-dup merge:", train_master["duplicate_group_id"].nunique())

In [ ]:
# Cell 3 — Merge near-duplicate group IDs into train_master
NEAR_DUP_GROUP_START = 900000  # well above any file-count-based ID, avoids collision

near_dup_pairs = list(near_dup_df[["file_a", "file_b"]].itertuples(index=False, name=None))

for i, (fa, fb) in enumerate(near_dup_pairs):
    group_id = NEAR_DUP_GROUP_START + i
    mask     = train_master["filename"].isin([fa, fb])
    matched  = train_master.loc[mask, "filename"].tolist()
    if len(matched) != 2:
        print(f"Warning: expected 2 matches for pair ({fa}, {fb}), found {matched}")
        continue
    train_master.loc[mask, "duplicate_group_id"] = group_id

print("Unique groups after near-dup merge:", train_master["duplicate_group_id"].nunique())

In [ ]:
# Cell 4 — Flag files excluded from CV and training
overlap_filenames = set(overlap_df["train_filename"])

# O_8873.jpg: confirmed mislabel — duplicate of R_799.jpg (cloth bag),
# true class = Recyclable, mislabeled as Organic. Excluded from training entirely.
MISLABEL_EXCLUDE = {"O_8873.jpg"}

train_master["exclude_from_cv"]       = train_master["filename"].isin(overlap_filenames)
train_master["exclude_from_training"] = train_master["filename"].isin(MISLABEL_EXCLUDE)

print("exclude_from_cv count      :", train_master["exclude_from_cv"].sum(),       "(expected 97)")
print("exclude_from_training count:", train_master["exclude_from_training"].sum(), "(expected 1)")

In [ ]:
# Cell 5 — Validate train_master integrity and save CSV
assert train_master["exclude_from_cv"].sum() == 97,      "Overlap count mismatch!"
assert train_master["exclude_from_training"].sum() == 1, "Mislabel exclude mismatch!"
assert train_master["filename"].duplicated().sum() == 0, "Duplicate filenames in master list!"

# No group should span multiple classes unless explicitly flagged as cross-class
cross_class_check = train_master.groupby("duplicate_group_id")["label"].nunique()
unexpected_cross  = set(cross_class_check[cross_class_check > 1].index)
flagged_groups    = set(
    groups_df.loc[groups_df["is_cross_class_group"] == True, "duplicate_group_id"] + offset
)
assert unexpected_cross.issubset(flagged_groups), (
    f"Unexpected cross-class groups: {unexpected_cross - flagged_groups}"
)

print("Total train files:", len(train_master))
print("Files available for CV:", len(train_master[
    (~train_master["exclude_from_cv"]) & (~train_master["exclude_from_training"])
]))
print("Cross-class groups detected:", len(unexpected_cross | flagged_groups),
      "-> all expected:", unexpected_cross.issubset(flagged_groups))

train_master.to_csv(output_dir / "train_master_with_groups.csv", index=False)
train_master.head(10)

In [ ]:
# Cell 6 — Visual sample grid per class
import random
from PIL import Image
import matplotlib.pyplot as plt

random.seed(42)

def sample_files_per_class(train_dir, class_folders, n_samples=500):
    samples = {}
    for cls in class_folders:
        files = list((train_dir / cls).rglob("*.*"))
        samples[cls] = random.sample(files, min(n_samples, len(files)))
    return samples

def show_sample_grid(sampled_files, class_folders, n_per_class=8):
    fig, axes = plt.subplots(len(class_folders), n_per_class,
                             figsize=(n_per_class * 2, len(class_folders) * 2.2))
    for row, cls in enumerate(class_folders):
        files_to_show = random.sample(sampled_files[cls], n_per_class)
        for col, f in enumerate(files_to_show):
            ax = axes[row, col]
            ax.imshow(Image.open(f).convert("RGB"))
            ax.axis("off")
        axes[row, 0].set_title(cls, loc="left", fontsize=11, fontweight="bold", x=-0.1, y=1.05)
    plt.tight_layout()
    plt.show()

class_folders = ["0_Recyclable", "1_Electronic", "2_Organic"]
sampled_files = sample_files_per_class(train_dir, class_folders, n_samples=500)
show_sample_grid(sampled_files, class_folders, n_per_class=8)

In [ ]:
# Cell 7 — Background complexity analysis per class
import numpy as np

# Estimate background complexity from corner patch variance
# Low variance -> plain/studio background | High variance -> complex/natural background
def compute_background_complexity(img, patch_size=30):
    arr = np.array(img)
    h, w, _ = arr.shape
    p = patch_size
    corners = [
        arr[0:p, 0:p],
        arr[0:p, w-p:w],
        arr[h-p:h, 0:p],
        arr[h-p:h, w-p:w],
    ]
    return np.mean([c.var() for c in corners])

bg_results = []
for cls, files in sampled_files.items():
    for f in files:
        try:
            img = Image.open(f).convert("RGB")
            bg_results.append({
                "class":       cls,
                "file":        str(f),
                "bg_variance": compute_background_complexity(img),
            })
        except Exception:
            pass

df_bg = pd.DataFrame(bg_results)

print("Background variance statistics per class:")
print(df_bg.groupby("class")["bg_variance"].describe())

threshold = 50
df_bg["is_plain_bg"] = df_bg["bg_variance"] < threshold
print("\nProportion of plain-background images (variance < 50) per class:")
print(df_bg.groupby("class")["is_plain_bg"].mean())

fig, ax = plt.subplots(figsize=(8, 5))
for cls in class_folders:
    subset = df_bg[df_bg["class"] == cls]["bg_variance"]
    ax.hist(subset, bins=40, alpha=0.5, label=cls)
ax.set_xlabel("Background Variance (4-corner patch)")
ax.set_ylabel("Frequency")
ax.set_title("Background Complexity Distribution per Class")
ax.legend()
plt.show()

In [ ]:
# Cell 8 — Image dimension, aspect ratio, and orientation analysis per class
dim_results = []
for cls, files in sampled_files.items():
    for f in files:
        try:
            with Image.open(f) as img:
                w, h = img.size
                dim_results.append({
                    "class":        cls,
                    "file":         str(f),
                    "width":        w,
                    "height":       h,
                    "aspect_ratio": round(w / h, 3),
                    "orientation":  "square" if abs(w - h) <= 5
                                    else ("landscape" if w > h else "portrait"),
                })
        except Exception:
            pass

df_dim = pd.DataFrame(dim_results)

print("Width & height statistics per class:")
print(df_dim.groupby("class")[["width", "height"]].describe())

print("\nAspect ratio statistics per class:")
print(df_dim.groupby("class")["aspect_ratio"].describe())

print("\nOrientation distribution per class (proportion):")
print(pd.crosstab(df_dim["class"], df_dim["orientation"], normalize="index").round(3))

In [ ]:
# Cell 9 — Electronic subgroup analysis: 150x150 icon vs larger images
electronic_dims = df_dim[df_dim["class"] == "1_Electronic"].copy()
electronic_dims["is_small_150"] = (
    (electronic_dims["width"] == 150) & (electronic_dims["height"] == 150)
)

n_small = electronic_dims["is_small_150"].sum()
n_total = len(electronic_dims)
print(f"Electronic images with exact 150x150 size: {n_small} / {n_total} ({n_small/n_total:.1%})")

print("\nWidth statistics for Electronic images that are NOT 150x150:")
print(electronic_dims[~electronic_dims["is_small_150"]]["width"].describe())

small_samples = electronic_dims[electronic_dims["is_small_150"]]["file"].head(5).tolist()
large_samples = (
    electronic_dims[~electronic_dims["is_small_150"]]
    .sort_values("width", ascending=False)["file"]
    .head(5).tolist()
)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, f in enumerate(small_samples):
    axes[0, i].imshow(Image.open(f))
    axes[0, i].set_title(f"150x150\n{Path(f).stem[:15]}", fontsize=8)
    axes[0, i].axis("off")
for i, f in enumerate(large_samples):
    axes[1, i].imshow(Image.open(f))
    axes[1, i].set_title(f"Large\n{Path(f).stem[:15]}", fontsize=8)
    axes[1, i].axis("off")
plt.suptitle("Electronic: 150x150 (top) vs Large (bottom)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 — Visual comparison: Recyclable vs Organic samples
random.seed(42)
recyclable_samples = random.sample(sampled_files["0_Recyclable"], 5)
organic_samples    = random.sample(sampled_files["2_Organic"], 5)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, f in enumerate(recyclable_samples):
    axes[0, i].imshow(Image.open(f))
    axes[0, i].set_title(f"Recyclable\n{Path(f).stem[:15]}", fontsize=8)
    axes[0, i].axis("off")

for i, f in enumerate(organic_samples):
    axes[1, i].imshow(Image.open(f))
    axes[1, i].set_title(f"Organic\n{Path(f).stem[:15]}", fontsize=8)
    axes[1, i].axis("off")

plt.suptitle("Sample Grid: Recyclable (top) vs Organic (bottom)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 — Brightness and per-channel color mean analysis per class
brightness_results = []

for cls, files in sampled_files.items():
    for f in files:
        try:
            with Image.open(f) as img:
                arr = np.array(img.convert("RGB"))
                brightness_results.append({
                    "class":      cls,
                    "file":       str(f),
                    "brightness": arr.mean(),
                    "r_mean":     arr[:, :, 0].mean(),
                    "g_mean":     arr[:, :, 1].mean(),
                    "b_mean":     arr[:, :, 2].mean(),
                })
        except Exception:
            pass

df_brightness = pd.DataFrame(brightness_results)

print("Brightness statistics per class:")
print(df_brightness.groupby("class")["brightness"].describe())

print("\nMean color channel (R/G/B) per class:")
print(df_brightness.groupby("class")[["r_mean", "g_mean", "b_mean"]].mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls in class_folders:
    subset = df_brightness[df_brightness["class"] == cls]["brightness"]
    axes[0].hist(subset, bins=40, alpha=0.5, label=cls)
axes[0].set_xlabel("Brightness (mean pixel value)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Brightness Distribution per Class")
axes[0].legend()

df_brightness.boxplot(column="brightness", by="class", ax=axes[1])
axes[1].set_title("Brightness per Class (Boxplot)")
axes[1].set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 — Assign fold indices via StratifiedGroupKFold
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5
SEED     = 42

cv_pool = train_master[
    (~train_master["exclude_from_cv"]) & (~train_master["exclude_from_training"])
].reset_index(drop=True)

print("CV pool size:", len(cv_pool))
print("Unique groups in CV pool:", cv_pool["duplicate_group_id"].nunique())

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

cv_pool["fold"] = -1

X      = cv_pool["filename"]
y      = cv_pool["label"]
groups = cv_pool["duplicate_group_id"]

for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
    cv_pool.loc[val_idx, "fold"] = fold_idx

assert (cv_pool["fold"] == -1).sum() == 0, "Some rows were not assigned a fold!"
print(cv_pool["fold"].value_counts().sort_index())

In [ ]:
# Cell 13 — Validate fold assignments: group integrity, class balance, fold size
# 1) No group should be split across multiple folds
group_fold_check = cv_pool.groupby("duplicate_group_id")["fold"].nunique()
leaky_groups     = group_fold_check[group_fold_check > 1]
assert len(leaky_groups) == 0, (
    f"GROUP LEAKAGE: {len(leaky_groups)} groups span multiple folds!\n{leaky_groups}"
)
print("Group integrity check: PASSED (no group spans multiple folds)")

# 2) Class balance per fold
balance             = cv_pool.groupby("fold")["label"].value_counts(normalize=True).unstack().round(4)
balance.columns     = ["Recyclable", "Electronic", "Organic"]
print(balance)

# Deviation from global class ratio
global_ratio        = cv_pool["label"].value_counts(normalize=True).sort_index()
global_ratio.index  = ["Recyclable", "Electronic", "Organic"]
deviation           = (balance - global_ratio).abs().round(4)
print("\nAbsolute deviation from global class ratio (global = {}):".format(
    global_ratio.round(4).to_dict()
))
print(deviation)

max_dev = deviation.values.max()
print(f"\nMax deviation across all folds/classes: {max_dev:.4f}")
if max_dev > 0.02:
    print("Warning: deviation > 2pp — flag before Phase 1 baseline training.")
else:
    print("OK: deviation within acceptable range (<2pp).")

# 3) Fold size check
print("\nFold sizes:")
print(cv_pool["fold"].value_counts().sort_index())

In [ ]:
# Cell 14 — Merge fold assignments into train_master and save
# Rows not in cv_pool (excluded files) get fold = -1
fold_map = dict(zip(cv_pool["filename"], cv_pool["fold"]))
train_master["fold"] = train_master["filename"].map(fold_map).fillna(-1).astype(int)

train_master.to_csv(output_dir / "train_master_with_folds.csv", index=False)
print("Saved train_master_with_folds.csv")
train_master["fold"].value_counts().sort_index()

In [ ]:
# Cell 15 — Load submission.csv and build test filepaths
submission_df = pd.read_csv(submission_path)
print(submission_df.shape)
print(submission_df.head())
print(submission_df.dtypes)

# Build filename from id column (test files are named "{id}.jpg")
submission_df["filename"] = submission_df["id"].astype(str) + ".jpg"
submission_df["filepath"] = submission_df["filename"].apply(lambda f: str(test_dir / f))

# Verify all expected test files exist on disk
missing = submission_df[~submission_df["filepath"].apply(lambda p: Path(p).exists())]
assert len(missing) == 0, f"Missing test files: {missing['filename'].tolist()}"

print(f"\nAll {len(submission_df)} test files found on disk, matching submission.csv order.")
submission_df.head()

In [ ]:
# Cell 16 — Audit image mode distribution across train and test
from PIL import Image

def get_image_mode(filepath):
    try:
        with Image.open(filepath) as img:
            return img.mode
    except Exception as e:
        return f"ERROR: {e}"

train_master["img_mode"]  = train_master["filepath"].apply(get_image_mode)
submission_df["img_mode"] = submission_df["filepath"].apply(get_image_mode)

print("Train mode distribution:")
print(train_master["img_mode"].value_counts())
print("\nTest mode distribution:")
print(submission_df["img_mode"].value_counts())

non_rgb_train = train_master[train_master["img_mode"] != "RGB"]
non_rgb_test  = submission_df[submission_df["img_mode"] != "RGB"]
print(f"\nNon-RGB train files: {len(non_rgb_train)} (expected 312: 293 Palette + 17 RGBA + 2 Grayscale)")
print(f"Non-RGB test files : {len(non_rgb_test)}")

In [ ]:
# Cell 17 — Define robust RGB image loader
def load_image_as_rgb(filepath):
    """
    Always returns a PIL Image in RGB mode.
    Handles RGBA (composites on white background to avoid black artifacts),
    Palette (P), Grayscale (L), and CMYK edge cases.
    """
    img = Image.open(filepath)

    if img.mode == "RGBA":
        # Composite onto white background — avoids black halos where alpha=0
        background = Image.new("RGB", img.size, (255, 255, 255))
        background.paste(img, mask=img.split()[3])
        img = background
    elif img.mode != "RGB":
        img = img.convert("RGB")

    return img

# Verify on a known RGBA sample
sample_non_rgb = train_master[train_master["img_mode"] == "RGBA"]
if len(sample_non_rgb) > 0:
    test_fp   = sample_non_rgb.iloc[0]["filepath"]
    converted = load_image_as_rgb(test_fp)
    print(f"Tested on: {test_fp}")
    print(f"Converted mode: {converted.mode}, size: {converted.size}")
else:
    print("No RGBA sample found in train_master.")

In [ ]:
# Cell 18 — Verify RGB conversion for all non-RGB modes
modes_to_test = ["P", "RGBA", "L"]

for mode in modes_to_test:
    sample = train_master[train_master["img_mode"] == mode]
    if len(sample) == 0:
        print(f"No {mode} sample found, skipping")
        continue
    fp = sample.iloc[0]["filepath"]
    try:
        converted = load_image_as_rgb(fp)
        assert converted.mode == "RGB", f"Conversion failed for {mode}: got {converted.mode}"
        print(f"OK: {mode} -> RGB | sample: {Path(fp).name} | size: {converted.size}")
    except Exception as e:
        print(f"Failed: {mode} conversion error: {e}")

# Verify all test Palette files convert cleanly
test_palette_samples = submission_df[submission_df["img_mode"] == "P"]
print(f"\nVerifying all {len(test_palette_samples)} test Palette files convert cleanly:")
for _, row in test_palette_samples.iterrows():
    converted = load_image_as_rgb(row["filepath"])
    assert converted.mode == "RGB"
print("OK: all test Palette files convert successfully.")

In [ ]:
# Cell 19 — WasteDataset
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

class WasteDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        """
        df        : DataFrame with at least ['filepath'] column,
                    and ['label'] if not is_test.
        transform : torchvision transform pipeline applied after RGB conversion.
        is_test   : if True, returns (image, filename) instead of (image, label).
        """
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = load_image_as_rgb(row["filepath"])

        if self.transform:
            img = self.transform(img)

        if self.is_test:
            return img, row["filename"]
        else:
            return img, int(row["label"])

In [ ]:
# Cell 20 — Define train and eval transforms
IMG_SIZE = 224  # baseline resolution for ConvNeXt V2 pretrained models

# Train: RandAugment + HFlip + limited ColorJitter + normalization
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.LANCZOS),
    T.RandomHorizontalFlip(p=0.5),
    T.RandAugment(num_ops=2, magnitude=7),
    T.ColorJitter(brightness=0.15, contrast=0.1, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Val/Test: deterministic, no augmentation
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.LANCZOS),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"Transforms ready. Baseline resolution: {IMG_SIZE}x{IMG_SIZE}")

In [ ]:
# Cell 21 — Build DataLoaders for fold 0 baseline
BATCH_SIZE   = 64  # ConvNeXt V2-Tiny @ 224px, 16GB VRAM baseline
NUM_WORKERS  = 4

fold_df          = pd.read_csv(output_dir / "train_master_with_folds.csv")
FOLD_TO_VALIDATE = 0

train_df = fold_df[(fold_df["fold"] != FOLD_TO_VALIDATE) & (fold_df["fold"] != -1)].reset_index(drop=True)
val_df   = fold_df[fold_df["fold"] == FOLD_TO_VALIDATE].reset_index(drop=True)

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")
print(f"Train label distribution:\n{train_df['label'].value_counts(normalize=True).sort_index()}")
print(f"Val label distribution:\n{val_df['label'].value_counts(normalize=True).sort_index()}")

train_dataset = WasteDataset(train_df, transform=train_transform, is_test=False)
val_dataset   = WasteDataset(val_df,   transform=eval_transform,  is_test=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Cell 22 — Sanity check batch shapes and pixel value range
images, labels = next(iter(train_loader))
print("Train batch images shape:", images.shape)   # expect [BATCH_SIZE, 3, 224, 224]
print("Train batch images dtype:", images.dtype)
print("Train batch labels shape:", labels.shape)
print("Train batch labels unique:", labels.unique())
print("Pixel value range (post-normalize):", images.min().item(), "to", images.max().item())

test_dataset = WasteDataset(submission_df, transform=eval_transform, is_test=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

test_images, test_filenames = next(iter(test_loader))
print("\nTest batch images shape:", test_images.shape)
print("Test batch filenames (first 5):", test_filenames[:5])

In [ ]:
# Cell 23 — Install timm and import training utilities
!pip install -q timm

import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"timm version    : {timm.__version__}")

In [ ]:
# Cell 24 — Build ConvNeXt V2-Tiny with 3-class head
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3

model = timm.create_model(
    "convnextv2_tiny.fcmae_ft_in22k_in1k",
    pretrained=True,
    num_classes=NUM_CLASSES,
)
model = model.to(DEVICE)

n_params    = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model      : convnextv2_tiny")
print(f"Total params    : {n_params:,}")
print(f"Trainable params: {n_trainable:,}")
print(f"Device          : {DEVICE}")

In [ ]:
# Cell 25 — Discriminative learning rate setup
BACKBONE_LR  = 1e-5
HEAD_LR      = 1e-4
WEIGHT_DECAY = 0.05

print("Model head structure:")
print(model.head)

# Split params: head vs backbone
head_params      = list(model.head.parameters())
head_param_ids   = {id(p) for p in head_params}
backbone_params  = [p for p in model.parameters() if id(p) not in head_param_ids]

optimizer = optim.AdamW([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": head_params,     "lr": HEAD_LR},
], weight_decay=WEIGHT_DECAY)

print(f"\nBackbone params : {sum(p.numel() for p in backbone_params):,} @ LR={BACKBONE_LR}")
print(f"Head params     : {sum(p.numel() for p in head_params):,} @ LR={HEAD_LR}")

In [ ]:
# Cell 26 — Loss, cosine scheduler with warmup, AMP scaler
NUM_EPOCHS          = 15
WARMUP_STEPS_RATIO  = 0.075  # 7.5% of total steps

# Plain CrossEntropy — no class weights, per Phase 1 lock
criterion    = nn.CrossEntropyLoss()

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_STEPS_RATIO)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + torch.cos(torch.tensor(progress * 3.14159265)).item())

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler    = GradScaler("cuda")

print(f"Loss        : CrossEntropyLoss (plain, no weights)")
print(f"Total steps : {total_steps}, Warmup steps: {warmup_steps} ({WARMUP_STEPS_RATIO*100:.1f}%)")
print(f"AMP scaler  : {scaler}")

In [ ]:
# Cell 27 — Training loop
from sklearn.metrics import f1_score
import numpy as np
import time
import glob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

# Attempt to resume from existing checkpoint
CHECKPOINT_SEARCH_PATHS = [
    "/kaggle/input/*/fold0-conv/*.pt",
    str(output_dir / "*.pt"),
]

found_checkpoints = []
for pattern in CHECKPOINT_SEARCH_PATHS:
    found_checkpoints.extend(glob.glob(pattern))

start_epoch  = 0
best_val_f1  = 0.0
best_epoch   = -1
history      = []

if found_checkpoints:
    def extract_score(path):
        try:
            return float(os.path.basename(path).split("_cv")[-1].replace(".pt", ""))
        except Exception:
            return -1.0

    best_checkpoint_path = max(found_checkpoints, key=extract_score)
    print(f"Checkpoint found: {best_checkpoint_path}")

    try:
        checkpoint = torch.load(best_checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        if "optimizer_state_dict" in checkpoint:
            try:
                optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            except Exception as opt_err:
                print(f"Optimizer state not restored: {opt_err}")
        start_epoch = checkpoint.get("epoch", 0)
        best_val_f1 = checkpoint.get("val_macro_f1", 0.0)
        best_epoch  = start_epoch
        print(f"Resumed from epoch {start_epoch}, val_macro_f1={best_val_f1:.4f}")
    except Exception as e:
        print(f"Failed to load checkpoint ({e}). Training from scratch.")
        start_epoch = 0
        best_val_f1 = 0.0
        best_epoch  = -1
else:
    print("No checkpoint found. Training from scratch.")

# Training loop
for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()

    # Train
    model.train()
    train_loss_sum = 0.0
    train_correct  = 0
    train_total    = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda"):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss_sum += loss.item() * images.size(0)
        preds           = outputs.argmax(dim=1)
        train_correct  += (preds == labels).sum().item()
        train_total    += images.size(0)

    train_loss = train_loss_sum / train_total
    train_acc  = train_correct / train_total

    # Validate
    model.eval()
    val_loss_sum   = 0.0
    val_preds_all  = []
    val_labels_all = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast("cuda"):
                outputs = model(images)
                loss    = criterion(outputs, labels)

            val_loss_sum += loss.item() * images.size(0)
            preds         = outputs.argmax(dim=1)
            val_preds_all.extend(preds.cpu().numpy().tolist())
            val_labels_all.extend(labels.cpu().numpy().tolist())

    val_loss        = val_loss_sum / len(val_labels_all)
    val_macro_f1    = f1_score(val_labels_all, val_preds_all, average="macro")
    val_f1_per_class = f1_score(val_labels_all, val_preds_all, average=None)
    epoch_time      = time.time() - epoch_start

    current_lr_backbone = optimizer.param_groups[0]["lr"]
    current_lr_head     = optimizer.param_groups[1]["lr"]

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_macro_f1={val_macro_f1:.4f} | "
        f"F1[Recy={val_f1_per_class[0]:.3f}, Elec={val_f1_per_class[1]:.3f}, Org={val_f1_per_class[2]:.3f}] | "
        f"LR[bb={current_lr_backbone:.2e}, head={current_lr_head:.2e}] | "
        f"time={epoch_time:.1f}s"
    )

    history.append({
        "epoch":             epoch + 1,
        "train_loss":        train_loss,
        "train_acc":         train_acc,
        "val_loss":          val_loss,
        "val_macro_f1":      val_macro_f1,
        "val_f1_recyclable": val_f1_per_class[0],
        "val_f1_electronic": val_f1_per_class[1],
        "val_f1_organic":    val_f1_per_class[2],
    })

    if val_macro_f1 > best_val_f1:
        best_val_f1   = val_macro_f1
        best_epoch    = epoch + 1
        checkpoint_path = os.path.join(output_dir, f"model_s9_baseline_cv{best_val_f1:.4f}.pt")
        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_macro_f1":         val_macro_f1,
            "val_f1_per_class":     val_f1_per_class.tolist(),
        }, checkpoint_path)
        print(f"  -> New best checkpoint saved: {checkpoint_path}")

print(f"\nTraining complete. Best val_macro_f1={best_val_f1:.4f} at epoch {best_epoch}")

# Save full epoch history
history_df  = pd.DataFrame(history)
print(history_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

history_csv = os.path.join(output_dir, "training_history_baseline.csv")
history_df.to_csv(history_csv, index=False)
print(f"\nHistory saved to {history_csv}")

In [ ]:
# Cell 28 — Electronic subpopulation sensitivity check: icon 150x150 vs natural photo
# Requires val_loader with shuffle=False and val_df in same order
model.eval()
val_preds_all  = []
val_labels_all = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast("cuda"):
            outputs = model(images)
        preds = outputs.argmax(dim=1)
        val_preds_all.extend(preds.cpu().numpy().tolist())
        val_labels_all.extend(labels.cpu().numpy().tolist())

val_filepaths_all = val_df["filepath"].tolist()
assert len(val_filepaths_all) == len(val_labels_all), (
    "Mismatch: val_df order must match val_loader (shuffle=False required)"
)

# Detect subpopulation by actual image dimensions
def get_image_dims(filepath):
    try:
        with Image.open(filepath) as img:
            return img.size
    except Exception:
        return (None, None)

val_dims        = [get_image_dims(fp) for fp in val_filepaths_all]
is_icon         = [(w == 150 and h == 150) for w, h in val_dims]

val_labels_np   = np.array(val_labels_all)
val_preds_np    = np.array(val_preds_all)
is_icon_np      = np.array(is_icon)
electronic_mask = (val_labels_np == 1)

icon_mask    = electronic_mask & is_icon_np
natural_mask = electronic_mask & ~is_icon_np

print(f"Total Electronic in val : {electronic_mask.sum()}")
print(f"  Icon (150x150)        : {icon_mask.sum()} ({icon_mask.sum()/electronic_mask.sum()*100:.1f}%)")
print(f"  Natural (other)       : {natural_mask.sum()} ({natural_mask.sum()/electronic_mask.sum()*100:.1f}%)")

icon_correct    = (val_preds_np[icon_mask] == 1).sum()
natural_correct = (val_preds_np[natural_mask] == 1).sum()
icon_acc        = icon_correct / icon_mask.sum() if icon_mask.sum() > 0 else float("nan")
natural_acc     = natural_correct / natural_mask.sum() if natural_mask.sum() > 0 else float("nan")

print(f"\nRecall per subpopulation:")
print(f"Icon (150x150) : {icon_acc:.4f} ({icon_correct}/{icon_mask.sum()})")
print(f"Natural photo  : {natural_acc:.4f} ({natural_correct}/{natural_mask.sum()})")

if not np.isnan(icon_acc) and not np.isnan(natural_acc):
    gap = abs(icon_acc - natural_acc)
    print(f"\nGap: {gap:.4f}")
    if gap > 0.05:
        print("Warning: gap > 5pp — model may be exploiting visual shortcuts (size/format).")
    else:
        print("OK: gap within acceptable range.")

In [ ]:
# Cell 29 — Grad-CAM shortcut check
!pip install grad-cam --quiet

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt

# Target last conv stage before head
target_layers = [model.stages[-1]]

def load_and_preprocess(filepath):
    img        = Image.open(filepath).convert("RGB")
    img_resized = img.resize((224, 224))
    img_np     = np.array(img_resized).astype(np.float32) / 255.0
    mean       = np.array([0.485, 0.456, 0.406])
    std        = np.array([0.229, 0.224, 0.225])
    img_norm   = (img_np - mean) / std
    tensor     = torch.from_numpy(img_norm.transpose(2, 0, 1)).unsqueeze(0).float().to(device)
    return img_np, tensor

# Build sample groups: shortcut risk vs control
def compute_bg_variance(filepath, patch_size=30):
    try:
        img = Image.open(filepath).convert("RGB")
        arr = np.array(img)
        h, w, _ = arr.shape
        corners = [
            arr[:patch_size, :patch_size],
            arr[:patch_size, -patch_size:],
            arr[-patch_size:, :patch_size],
            arr[-patch_size:, -patch_size:],
        ]
        return np.mean([c.var() for c in corners])
    except Exception:
        return None

recyclable_indices = np.where(val_labels_np == 0)[0]
electronic_indices = np.where(val_labels_np == 1)[0]

recyclable_bg_var  = {
    idx: compute_bg_variance(val_filepaths_all[idx])
    for idx in recyclable_indices[:200]
}
recyclable_plain   = [idx for idx, v in recyclable_bg_var.items() if v is not None and v < 50]
recyclable_complex = [idx for idx, v in recyclable_bg_var.items() if v is not None and v >= 50]
electronic_icon    = [idx for idx in electronic_indices if is_icon_np[idx]]
electronic_natural = [idx for idx in electronic_indices if not is_icon_np[idx]]

np.random.seed(42)
sample_groups = {
    "Recyclable_PlainBG (shortcut risk)":  np.random.choice(recyclable_plain,   min(5, len(recyclable_plain)),   replace=False),
    "Recyclable_ComplexBG (control)":      np.random.choice(recyclable_complex, min(3, len(recyclable_complex)), replace=False),
    "Electronic_Icon150 (shortcut risk)":  np.random.choice(electronic_icon,    min(5, len(electronic_icon)),    replace=False),
    "Electronic_Natural (control)":        np.random.choice(electronic_natural, min(3, len(electronic_natural)), replace=False),
}

print("Sample counts per group:")
for group_name, indices in sample_groups.items():
    print(f"  {group_name}: {len(indices)} samples")

# Run Grad-CAM
cam = GradCAM(model=model, target_layers=target_layers)

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for row_idx, (group_name, indices) in enumerate(sample_groups.items()):
    for col_idx in range(5):
        ax = axes[row_idx, col_idx]
        if col_idx < len(indices):
            idx        = indices[col_idx]
            filepath   = val_filepaths_all[idx]
            true_label = val_labels_np[idx]
            pred_label = val_preds_np[idx]
            img_np, tensor = load_and_preprocess(filepath)
            grayscale_cam  = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(int(pred_label))])[0]
            cam_image      = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
            ax.imshow(cam_image)
            ax.set_title(f"true={true_label} pred={pred_label}", fontsize=9)
        ax.axis("off")
    axes[row_idx, 0].set_ylabel(group_name, fontsize=10, rotation=0, ha="right", va="center")

plt.tight_layout()
plt.savefig(str(output_dir / "gradcam_shortcut_check.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {output_dir / 'gradcam_shortcut_check.png'}")

In [ ]:
# Cell 30 — Grad-CAM Electronic failure mode: FP and FN cases
# False Positive: predicted=1, true != 1
fp_electronic_mask    = (val_preds_np == 1) & (val_labels_np != 1)
fp_electronic_indices = np.where(fp_electronic_mask)[0]

# False Negative: true=1, predicted != 1
fn_electronic_mask    = (val_labels_np == 1) & (val_preds_np != 1)
fn_electronic_indices = np.where(fn_electronic_mask)[0]

print(f"False Positive Electronic (predicted=1, true!=1): {len(fp_electronic_indices)} cases")
for idx in fp_electronic_indices:
    print(f"  idx={idx}, true={val_labels_np[idx]}, filepath={val_filepaths_all[idx]}")

print(f"\nFalse Negative Electronic (true=1, predicted!=1): {len(fn_electronic_indices)} cases")
for idx in fn_electronic_indices:
    print(f"  idx={idx}, pred={val_preds_np[idx]}, filepath={val_filepaths_all[idx]}")

recy_org_confusion_mask    = (
    ((val_labels_np == 0) & (val_preds_np == 2)) |
    ((val_labels_np == 2) & (val_preds_np == 0))
)
recy_org_confusion_indices = np.where(recy_org_confusion_mask)[0]
print(f"\nRecyclable <-> Organic confusion: {len(recy_org_confusion_indices)} cases")

# Visualize FP + FN with Grad-CAM
fp_fn_indices = list(fp_electronic_indices) + list(fn_electronic_indices)
fp_fn_labels  = ["FP"] * len(fp_electronic_indices) + ["FN"] * len(fn_electronic_indices)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, (idx, err_type) in enumerate(zip(fp_fn_indices, fp_fn_labels)):
    row, col = divmod(i, 5)
    ax       = axes[row, col]

    filepath   = val_filepaths_all[idx]
    true_label = val_labels_np[idx]
    pred_label = val_preds_np[idx]

    img_np, tensor = load_and_preprocess(filepath)
    grayscale_cam  = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(int(pred_label))])[0]
    cam_image      = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

    ax.imshow(cam_image)
    ax.set_title(f"{err_type} | true={true_label} pred={pred_label}", fontsize=9)
    ax.axis("off")

for i in range(len(fp_fn_indices), 10):
    row, col = divmod(i, 5)
    axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(str(output_dir / "gradcam_electronic_errors.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {output_dir / 'gradcam_electronic_errors.png'}")

In [ ]:
# Cell 31 — Manual mislabel candidate verification
import math

mislabel_candidates = [
    {"filename": "R_3825.jpg",     "current_label": "Recyclable", "suspected": "Electronic (laptop)"},
    {"filename": "R_3733.jpg",     "current_label": "Recyclable", "suspected": "Electronic (laptop)"},
    {"filename": "O_7776.jpg",     "current_label": "Organic",    "suspected": "Electronic (panel)"},
    {"filename": "battery_61.jpg", "current_label": "Electronic", "suspected": "Recyclable (bottle)"},
]

fig, axes = plt.subplots(1, len(mislabel_candidates), figsize=(20, 5))
for ax, cand in zip(axes, mislabel_candidates):
    found_path = None
    for class_folder in class_folders:
        candidate_path = train_dir / class_folder / cand["filename"]
        if candidate_path.exists():
            found_path = str(candidate_path)
            break
    if found_path:
        img = load_image_as_rgb(found_path)
        ax.imshow(img)
        ax.set_title(
            f"{cand['filename']}\nCurrent: {cand['current_label']}\nSuspected: {cand['suspected']}",
            fontsize=9,
        )
    else:
        ax.text(0.5, 0.5, f"File not found:\n{cand['filename']}",
                ha="center", va="center", fontsize=10, color="red")
    ax.axis("off")

plt.tight_layout()
plt.savefig(output_dir / "mislabel_verification_batch1.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {output_dir / 'mislabel_verification_batch1.png'}")

# Find FN Electronic -> Organic candidates for excavator filename lookup
print("\nSearching for FN candidates (true=Electronic, predicted=Organic):")
fn_electronic_as_organic = val_df[
    (val_df["label"] == 1) &
    (val_df.index.isin([
        i for i, (true, pred) in enumerate(zip(val_labels_all, val_preds_all))
        if true == 1 and pred == 2
    ]))
]
print(f"Total FN (Electronic->Organic) in val set: {len(fn_electronic_as_organic)}")
print(fn_electronic_as_organic[["filename", "filepath"]].to_string(index=False))

In [ ]:
# Cell 32 — Non-standard filename pattern analysis
import re

def is_standard_pattern(filename):
    patterns = [
        r"^R_\d+\.(jpg|jpeg|png)$",
        r"^O_\d+\.(jpg|jpeg|png)$",
        r"^[A-Za-z]+_\d+(\(\d+\))?\.(jpg|jpeg|png)$",
        r"^(Copy of )?IMG_\d+.*\.(jpg|jpeg|png)$",
        r"^\d+\.(jpg|jpeg|png)$",
    ]
    return any(re.match(p, filename, re.IGNORECASE) for p in patterns)

def looks_like_caption(filename):
    word_count     = len(re.findall(r"[a-zA-Z]{3,}", filename))
    has_separator  = "-" in filename or " " in filename
    return word_count >= 4 and has_separator

suspicious = train_master[~train_master["filename"].apply(is_standard_pattern)].copy()
suspicious["looks_like_caption"] = suspicious["filename"].apply(looks_like_caption)

print(f"Total non-standard files (all classes): {len(suspicious)}")
print(f"Caption-like filenames                : {suspicious['looks_like_caption'].sum()}")
print(suspicious.groupby("label").size())

In [ ]:
# Cell 33 — Caption-like candidate visual inspection
caption_subset = suspicious[suspicious["looks_like_caption"]].copy()

print("Caption-like breakdown by label:")
print(caption_subset.groupby("label").size())

caption_subset[["filename", "label"]].to_csv(
    output_dir / "caption_like_candidates.csv", index=False
)
print(f"\nSaved: {output_dir / 'caption_like_candidates.csv'}")

# Stratified sample for visual check
N_PER_CLASS  = 10
sample_list  = []
for cls, group in caption_subset.groupby("label"):
    n = min(N_PER_CLASS, len(group))
    sample_list.append(group.sample(n=n, random_state=42))
visual_sample = pd.concat(sample_list).reset_index(drop=True)

print(f"\nTotal samples for visual check: {len(visual_sample)}")

n_cols = 5
n_rows = math.ceil(len(visual_sample) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for idx, row in visual_sample.iterrows():
    filepath = train_dir / str(row["label"]) / row["filename"]
    try:
        img = load_image_as_rgb(str(filepath))
        axes[idx].imshow(img)
        axes[idx].set_title(f"{row['filename'][:30]}\nLabel: {row['label']}", fontsize=8)
    except Exception as e:
        axes[idx].set_title(f"Error: {e}", fontsize=8)
    axes[idx].axis("off")

for idx in range(len(visual_sample), len(axes)):
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig(output_dir / "caption_visual_check_sample.png", dpi=100)
plt.show()
print(f"Saved: {output_dir / 'caption_visual_check_sample.png'}")